# Parameter Sensitivity Analysis

Supplementary figure showing scaffold fingerprinting robustness across:
1. **Parcellation granularity:** AAL89, Schaefer100, Shen268, Schaefer300, Glasser379
2. **FC threshold:** 100% (full), 25%, 1% of edges  
3. **Scaffold vs FC** comparison across all parcellations

Data: HCP 100 Unrelated (N=100), Schaefer 300 no high-pass variant.

In [ ]:
import sys
import sys
sys.path.append("..")
from topofinger import config

import os
import numpy as np
import pickle as pk
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import OrderedDict
from pingouin import compute_effsize

from topofinger.fingerprinting import compute_idiff, compute_effect_size, compute_success_rate

%load_ext autoreload
%autoreload 2

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

## 1. Load All Parcellation Data

In [ ]:
# --- Configuration ---
BASE = config.HCP_100 + ""

parcellations = OrderedDict([
    ("AAL 90",      {"dir": "aal89",      "n_rois": 90}),
    ("Schaefer 100", {"dir": "schaefer100", "n_rois": 100}),
    ("Shen 268",    {"dir": "shen268",    "n_rois": 268}),
    ("Schaefer 300", {"dir": "schaefer300/no_hp", "n_rois": 300}),
    ("Glasser 360", {"dir": "glasser379", "n_rois": 360, "suffix": "_cortex"}),  # 360 cortical regions
])

# --- Load scaffold similarity matrices ---
scaffold_results = {}
for name, info in parcellations.items():
    sim_path = os.path.join(BASE, info["dir"], "scaffolds", "whole", f"scaffold_correlations_whole{info.get('suffix', '')}.npy")
    if os.path.exists(sim_path):
        sim = np.load(sim_path)
        scaffold_results[name] = sim
        print(f"✓ {name}: {sim.shape[0]}×{sim.shape[1]} scaffold similarity matrix")
    else:
        print(f"✗ {name}: not found at {sim_path}")

# --- Load FC similarity matrices (th=1%, th=25%, th=100%) ---
fc_results = {}
for name, info in parcellations.items():
    fc_results[name] = {}
    for th in [1, 25, 100]:
        fc_path = os.path.join(BASE, info["dir"], "fc", "whole", f"fc_correlations_whole_th{th}{info.get('suffix', '')}.npy")
        if os.path.exists(fc_path):
            sim = np.load(fc_path)
            fc_results[name][th] = sim
            print(f"  FC {name} th{th}: {sim.shape[0]}×{sim.shape[1]}")

print(f"\nLoaded scaffold for {len(scaffold_results)} parcellations")

## 2. Compute Fingerprinting Metrics

In [ ]:
# --- Scaffold metrics ---
scaffold_metrics = {}
for name, sim in scaffold_results.items():
    idiff, i_self, i_others = compute_idiff(sim)
    d = compute_effect_size(sim)
    sr1, sr2 = compute_success_rate(sim)
    scaffold_metrics[name] = {
        "cohen_d": d, "sr_mean": (sr1 + sr2) / 2,
        "idiff": idiff, "i_self": i_self, "i_others": i_others,
        "n_rois": parcellations[name]["n_rois"],
    }

# --- FC metrics (multiple thresholds) ---
fc_metrics = {}
for name in fc_results:
    fc_metrics[name] = {}
    for th, sim in fc_results[name].items():
        idiff, i_self, i_others = compute_idiff(sim)
        d = compute_effect_size(sim)
        sr1, sr2 = compute_success_rate(sim)
        fc_metrics[name][th] = {
            "cohen_d": d, "sr_mean": (sr1 + sr2) / 2,
            "idiff": idiff, "i_self": i_self, "i_others": i_others,
        }

# Print summary
print(f"{'Parcellation':<16} {'ROIs':>5} {'Scaff d':>9} {'Scaff SR':>9} | {'FC100 d':>8} {'FC25 d':>8} {'FC1 d':>8}")
print("-" * 80)
for name in scaffold_metrics:
    sm = scaffold_metrics[name]
    row = f"{name:<16} {sm['n_rois']:>5} {sm['cohen_d']:>9.2f} {sm['sr_mean']:>9.1f} |"
    for th in [100, 25, 1]:
        if name in fc_metrics and th in fc_metrics[name]:
            row += f" {fc_metrics[name][th]['cohen_d']:>8.2f}"
        else:
            row += f" {'N/A':>8}"
    print(row)

## 3. Main Sensitivity Figure

In [ ]:
# --- Supplementary Figure: Parameter Sensitivity ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

parc_names = list(scaffold_metrics.keys())
n_rois = [scaffold_metrics[p]["n_rois"] for p in parc_names]
scaffold_d = [scaffold_metrics[p]["cohen_d"] for p in parc_names]
scaffold_sr = [scaffold_metrics[p]["sr_mean"] for p in parc_names]

# --- Panel A: Cohen's d vs Parcellation Granularity ---
ax = axes[0]
# Scaffold
ax.plot(n_rois, scaffold_d, 'o-', color='#E53935', linewidth=2, markersize=10, label='Scaffold', zorder=3)
# FC at different thresholds
for th, color, ls in [(100, '#1565C0', '-'), (25, '#42A5F5', '--'), (1, '#90CAF9', ':')]:
    fc_d = []
    fc_rois = []
    for p in parc_names:
        if p in fc_metrics and th in fc_metrics[p]:
            fc_d.append(fc_metrics[p][th]["cohen_d"])
            fc_rois.append(parcellations[p]["n_rois"])
    if fc_d:
        ax.plot(fc_rois, fc_d, 's-', color=color, linewidth=1.5, markersize=7, 
                linestyle=ls, label=f'FC (top {th}%)', alpha=0.8)

ax.set_xlabel("Number of ROIs", fontsize=13)
ax.set_ylabel("Cohen's d", fontsize=13)
ax.set_title("A) Effect Size vs Parcellation", fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_xticks(n_rois)
ax.set_xticklabels([str(r) for r in n_rois], fontsize=10)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4)
ax.grid(alpha=0.3)

# --- Panel B: Success Rate vs Parcellation ---
ax = axes[1]
ax.plot(n_rois, scaffold_sr, 'o-', color='#E53935', linewidth=2, markersize=10, label='Scaffold', zorder=3)
for th, color, ls in [(100, '#1565C0', '-'), (25, '#42A5F5', '--'), (1, '#90CAF9', ':')]:
    fc_sr = []
    fc_rois = []
    for p in parc_names:
        if p in fc_metrics and th in fc_metrics[p]:
            fc_sr.append(fc_metrics[p][th]["sr_mean"])
            fc_rois.append(parcellations[p]["n_rois"])
    if fc_sr:
        ax.plot(fc_rois, fc_sr, 's-', color=color, linewidth=1.5, markersize=7, 
                linestyle=ls, label=f'FC (top {th}%)', alpha=0.8)

ax.set_xlabel("Number of ROIs", fontsize=13)
ax.set_ylabel("Success Rate (%)", fontsize=13)
ax.set_title("B) Success Rate vs Parcellation", fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xticks(n_rois)
ax.set_xticklabels([str(r) for r in n_rois], fontsize=10)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)

# --- Panel C: Scaffold advantage (ratio scaffold_d / FC_d) ---
ax = axes[2]
for th, color, off in [(100, '#1565C0', -0.27), (25, '#42A5F5', 0.0), (1, '#90CAF9', 0.27)]:
    ratios = []
    rois_th = []
    for p in parc_names:
        if p in fc_metrics and th in fc_metrics[p]:
            ratio = scaffold_metrics[p]["cohen_d"] / fc_metrics[p][th]["cohen_d"]
            ratios.append(ratio)
            rois_th.append(parcellations[p]["n_rois"])
    if ratios:
        ax.bar([r + off for r in range(len(rois_th))], ratios, 
               width=0.25, color=color, label=f'vs FC top {th}%', edgecolor='black', linewidth=0.5)

ax.set_xlabel("Parcellation", fontsize=13)
ax.set_ylabel("Scaffold d / FC d", fontsize=13)
ax.set_title("C) Scaffold Advantage Ratio", fontsize=14, fontweight='bold')
ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Parity (ratio=1)')
ax.set_xticks(range(len(parc_names)))
ax.set_xticklabels([p.split()[0] + "\n" + p.split()[1] for p in parc_names], fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
os.makedirs(config.FIGURES_RAW + "/supplementary", exist_ok=True)
plt.savefig(config.FIGURES_RAW + "/supplementary/parameter_sensitivity.png", dpi=300, bbox_inches='tight')
plt.savefig(config.FIGURES_RAW + "/supplementary/parameter_sensitivity.pdf", bbox_inches='tight')
plt.show()

print("Saved to " + config.FIGURES_RAW + "/supplementary/parameter_sensitivity.{png,pdf}")

## 4. FC Threshold Sensitivity Heatmap

In [ ]:
# --- Heatmap: Cohen's d for scaffold + FC thresholds across parcellations ---

methods = ['Scaffold'] + [f'FC top {th}%' for th in [100, 25, 1]]
heatmap_data = np.full((len(methods), len(parc_names)), np.nan)

for j, name in enumerate(parc_names):
    heatmap_data[0, j] = scaffold_metrics[name]["cohen_d"]
    for i, th in enumerate([100, 25, 1]):
        if name in fc_metrics and th in fc_metrics[name]:
            heatmap_data[i + 1, j] = fc_metrics[name][th]["cohen_d"]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[p.replace(' ', '\n') for p in parc_names],
            yticklabels=methods,
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label': "Cohen's d"})
ax.set_title("Fingerprinting Effect Size: Scaffold vs FC Across Parcellations and Thresholds",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(config.FIGURES_RAW + "/supplementary/sensitivity_heatmap.png", dpi=300, bbox_inches='tight')
plt.savefig(config.FIGURES_RAW + "/supplementary/sensitivity_heatmap.pdf", bbox_inches='tight')
plt.show()

## 5. Filtering × Parcellation Interaction (1200 Subjects)

For the 1200-subject dataset, only Schaefer 300 has scaffolds across filtering variants.
Show filtering results alongside the parcellation sensitivity from 100 unrelated.

In [ ]:
# --- Combined summary: Filtering (1200subj) + Parcellation (100unrel) ---

# Load 1200-subject filtering results
filter_labels = {
    "NoGSR_NoFilt": "No Filter",
    "NoGSR_LowPassOnly": "Low-Pass",
    "NoGSR_HighPassOnly": "High-Pass",
    "NoGSR_Bandpass": "Bandpass",
}

filter_metrics = {}
for variant, label in filter_labels.items():
    sim_path = f"{config.HCP_1200}/{variant}/scaffolds/whole/scaffold_correlations_whole_cortex.npy"
    if os.path.exists(sim_path):
        sim = np.load(sim_path)
        d = compute_effect_size(sim)
        sr1, sr2 = compute_success_rate(sim)
        filter_metrics[label] = {"cohen_d": d, "sr_mean": (sr1 + sr2) / 2, "n": sim.shape[0]}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Parcellation sensitivity (100unrel) ---
ax = axes[0]
colors_parc = ['#E53935'] * len(parc_names)
bars = ax.bar(range(len(parc_names)), scaffold_d, color=colors_parc, 
              edgecolor='black', linewidth=0.5, alpha=0.85)
for bar, val in zip(bars, scaffold_d):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(parc_names)))
ax.set_xticklabels([p.replace(' ', '\n') for p in parc_names], fontsize=10)
ax.set_ylabel("Cohen's d", fontsize=13)
ax.set_title("Parcellation Sensitivity\n(HCP 100 Unrelated, Scaffold)", fontsize=13, fontweight='bold')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4)
ax.grid(alpha=0.3, axis='y')

# --- Right: Filtering sensitivity (1200subj) ---
ax = axes[1]
flabels = list(filter_metrics.keys())
fvals = [filter_metrics[l]["cohen_d"] for l in flabels]
colors_filt = ['#FF9800'] * len(flabels)
bars = ax.bar(range(len(flabels)), fvals, color=colors_filt,
              edgecolor='black', linewidth=0.5, alpha=0.85)
for bar, val in zip(bars, fvals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(flabels)))
ax.set_xticklabels(flabels, fontsize=10)
ax.set_ylabel("Cohen's d", fontsize=13)
ax.set_title("Filtering Sensitivity\n(HCP ~1005, Schaefer 300, Scaffold)", fontsize=13, fontweight='bold')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(config.FIGURES_RAW + "/supplementary/sensitivity_combined.png", dpi=300, bbox_inches='tight')
plt.savefig(config.FIGURES_RAW + "/supplementary/sensitivity_combined.pdf", bbox_inches='tight')
plt.show()

# Print combined summary
print("\n" + "="*60)
print("PARAMETER SENSITIVITY SUMMARY")
print("="*60)
print("\nParcellation sensitivity (100 Unrelated):")
for name in parc_names:
    sm = scaffold_metrics[name]
    print(f"  {name:<16} d={sm['cohen_d']:.2f}  SR={sm['sr_mean']:.1f}%")
print(f"\n  Range of Cohen's d: {min(scaffold_d):.2f} — {max(scaffold_d):.2f} (Δ = {max(scaffold_d)-min(scaffold_d):.2f})")

print("\nFiltering sensitivity (1005 Subjects):")
for label, m in filter_metrics.items():
    print(f"  {label:<16} d={m['cohen_d']:.2f}  SR={m['sr_mean']:.1f}%")
print(f"\n  Range of Cohen's d: {min(fvals):.2f} — {max(fvals):.2f} (Δ = {max(fvals)-min(fvals):.2f})")